In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random

class CountingBlackjackEnv(gym.Env):
    def __init__(self):
        super().__init__()
        
        self.action_space = spaces.Discrete(3)
        
        self.observation_space = spaces.Box(
            low=np.array([4, 1, 0, -20]), 
            high=np.array([31, 11, 1, 20]), 
            dtype=np.float32
        )

        self.num_decks = 6
        self.reshuffle_threshold = 2 * 52
        self.shoe = []
        self.running_count = 0
        self._build_shoe()
    
    def _build_shoe(self):
        self.shoe = [2,3,4,5,6,7,8,9,10,10,10,10,11] * 4 * self.num_decks
        random.shuffle(self.shoe)
        self.running_count = 0

    def _draw_card(self):
        if len(self.shoe) < self.reshuffle_threshold:
            self._build_shoe()
        card = self.shoe.pop()
        self.update_count(card)
        return card
    
    def update_count(self, card):
        if card in [2, 3, 4, 5, 6]:
            self.running_count += 1
        elif card in [10, 11]:  
            self.running_count -= 1
    
    def _calculate_true_count(self):
        remaining_decks = len(self.shoe) / 52
        if remaining_decks > 0:
            return self.running_count / remaining_decks
        else:   
            return 0  

    
    def _calculate_score(self, hand):
        score = sum(hand)
        aces = hand.count(11)
        while score > 21 and aces > 0:
            score -= 10
            aces -= 1
        return score

    def _has_usable_ace(self, hand):
        return 11 in hand and sum(hand) <= 21
    
    def _get_obs(self):
        """
        Przekształca wewnętrzny stan gry na wektor (tablicę) zrozumiałą dla sieci neuronowej.
        """
        player_score = self._calculate_score(self.player_hand)
        dealer_card = self.dealer_hand[0] 
        
        usable_ace = int(self._has_usable_ace(self.player_hand))
        true_count = float(self._calculate_true_count())
        
        return np.array([player_score, dealer_card, usable_ace, true_count], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.player_hand = [self._draw_card(), self._draw_card()]
        self.dealer_hand = [self._draw_card(), self._draw_card()]
        
        obs = self._get_obs()
        
        info = {}
        
        return obs, info

    def step(self, action):
        reward = 0.0
        terminated = False
        truncated = False
        info = {}
        if action == 2:  # DOUBLE
            self.player_hand.append(self._draw_card())
            terminated = True
            if self._calculate_score(self.player_hand) > 21:
                reward = -2.0
        elif action == 1:
            self.player_hand.append(self._draw_card())
            if self._calculate_score(self.player_hand) > 21:
                reward = -1.0  # Bust - przegrana
                terminated = True
        elif action == 0:
            terminated = True

        player_score = self._calculate_score(self.player_hand)
        if terminated and player_score <= 21:

            while self._calculate_score(self.dealer_hand) < 17:
                self.dealer_hand.append(self._draw_card())
            dealer_score = self._calculate_score(self.dealer_hand)
            multiplier = 2.0 if action == 2 else 1.0  # Podwójna stawka za DOUBLE

            if dealer_score > 21:
                reward = 1.0 * multiplier  # Krupier bust
            elif player_score > dealer_score:
                reward = 1.0 * multiplier  # Gracz ma więcej
            elif player_score < dealer_score:
                reward = -1.0 * multiplier # Krupier ma więcej
            else:
                reward = 0.0  # Remis

           
        
        if terminated:
            info['final_player_hand'] = self.player_hand
            info['final_dealer_hand'] = self.dealer_hand
        
        return self._get_obs(), reward, terminated, truncated, info

In [ ]:
from stable_baselines3 import DQN

env = CountingBlackjackEnv()

# model = DQN("MlpPolicy", 
#             env, 
#             verbose=1,
#             device="cpu", 
#             buffer_size=10000,
#             exploration_fraction=0.4,
#             optimize_memory_usage=True,
#             replay_buffer_kwargs={"handle_timeout_termination": False})

# model = DQN(
#     "MlpPolicy", 
#     env, 
#     verbose=1,
#     learning_rate=5e-4,          
#     buffer_size=50000,
#     learning_starts=2000, 
#     batch_size=64,               
#     exploration_fraction=0.4,    
#     exploration_final_eps=0.02,  
#     target_update_interval=1000, 
#     replay_buffer_kwargs={"handle_timeout_termination": False}
# )

model = DQN(
    "MlpPolicy", 
    env, 
    verbose=0,
    learning_rate=3e-4,          
    buffer_size=100000,          
    learning_starts=5000, 
    batch_size=256,              
    exploration_fraction=0.5,    
    exploration_final_eps=0.01,  
    target_update_interval=1000,
    policy_kwargs=dict(net_arch=[128, 128]), 
    replay_buffer_kwargs={"handle_timeout_termination": False}
)

model.learn(total_timesteps=5000000, progress_bar=True)

model.save("models/dqn_blackjack_bot")

Output()

In [ ]:

# model = DQN.load("twoj_zapisany_model")
# env = TwojeSrodowiskoBlackjack()

def evaluate_blackjack_model(model, env, num_episodes=10000):
    wins = 0
    losses = 0
    draws = 0
    total_reward = 0.0

    print(f"Rozpoczynam testowanie modelu na {num_episodes} partiach...")

    for _ in range(num_episodes):
        obs, info = env.reset()
        done = False
        
        while not done:
            action, _states = model.predict(obs, deterministic=True)
            
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

        total_reward += reward
        if reward > 0:
            wins += 1
        elif reward < 0:
            losses += 1
        else:
            draws += 1

    win_rate = (wins / num_episodes) * 100
    loss_rate = (losses / num_episodes) * 100
    draw_rate = (draws / num_episodes) * 100
    avg_reward = total_reward / num_episodes

    print("\n=== WYNIKI TESTU ===")
    print(f"Rozegrane partie: {num_episodes}")
    print(f"Wygrane gracza: {wins} ({win_rate:.2f}%)")
    print(f"Przegrane (Wygrane kasyna): {losses} ({loss_rate:.2f}%)")
    print(f"Remisy: {draws} ({draw_rate:.2f}%)")
    print(f"Średnia nagroda na partię: {avg_reward:.4f}")
    
    return win_rate, loss_rate, draw_rate

evaluate_blackjack_model(model, env, num_episodes=10000)

Rozpoczynam testowanie modelu na 10000 partiach...

=== WYNIKI TESTU ===
Rozegrane partie: 10000
Wygrane gracza: 4057 (40.57%)
Przegrane (Wygrane kasyna): 4940 (49.40%)
Remisy: 1003 (10.03%)
Średnia nagroda na partię: -0.0575


(40.57, 49.4, 10.03)

In [ ]:
import pandas as pd

def analyze_bot_strategy(model):
    player_sums = range(8, 20)
    dealer_cards = range(2, 12) 
    usable_ace = 0 

    strategy_map = []

    for player_score in player_sums:
        row = []
        for dealer_card in dealer_cards:
            obs = np.array([player_score, dealer_card, usable_ace, 0.0], dtype=np.float32)
            
            action, _ = model.predict(obs, deterministic=True)
            
            action_symbol = {0: "S", 1: "H", 2: "D"}.get(int(action))
            row.append(action_symbol)
        strategy_map.append(row)

    df = pd.DataFrame(
        strategy_map, 
        columns=[f"D:{c}" for c in dealer_cards], 
        index=[f"P:{s}" for s in player_sums]
    )
    
    print("\n=== TABELA STRATEGII BOTA (S=Stand, H=Hit, D=Double) ===")
    print("Analiza dla rąk bez użytecznego asa:")
    print(df)
    return df

analyze_bot_strategy(model)


=== TABELA STRATEGII BOTA (S=Stand, H=Hit, D=Double) ===
Analiza dla rąk bez użytecznego asa:
     D:2 D:3 D:4 D:5 D:6 D:7 D:8 D:9 D:10 D:11
P:8    H   H   H   H   H   H   H   H    H    H
P:9    D   D   D   D   D   H   H   H    H    H
P:10   D   D   D   D   D   D   D   H    H    H
P:11   D   D   D   D   D   D   D   D    D    H
P:12   H   H   H   H   H   H   H   H    H    H
P:13   H   H   H   H   H   H   H   H    H    H
P:14   H   H   H   H   H   H   H   H    H    H
P:15   H   H   H   S   H   H   H   H    H    H
P:16   S   S   S   S   S   H   H   H    H    H
P:17   S   S   S   S   S   S   S   S    S    H
P:18   S   S   S   S   S   S   S   S    S    S
P:19   S   S   S   S   S   S   S   S    S    S


,D:2,D:3,D:4,D:5,D:6,D:7,D:8,D:9,D:10,D:11
P:8,H,H,H,H,H,H,H,H,H,H
P:9,D,D,D,D,D,H,H,H,H,H
P:10,D,D,D,D,D,D,D,H,H,H
P:11,D,D,D,D,D,D,D,D,D,H
P:12,H,H,H,H,H,H,H,H,H,H
P:13,H,H,H,H,H,H,H,H,H,H
P:14,H,H,H,H,H,H,H,H,H,H
P:15,H,H,H,S,H,H,H,H,H,H
P:16,S,S,S,S,S,H,H,H,H,H
P:17,S,S,S,S,S,S,S,S,S,H


In [ ]:
import pandas as pd
import numpy as np

def analyze_bot_strategy(model, true_count=0.0):
    player_sums = range(8, 20)
    dealer_cards = range(2, 12) 
    usable_ace = 0 

    strategy_map = []

    for player_score in player_sums:
        row = []
        for dealer_card in dealer_cards:
            obs = np.array([player_score, dealer_card, usable_ace, true_count], dtype=np.float32)
            
            action, _ = model.predict(obs, deterministic=True)
            
            action_symbol = {0: "S", 1: "H", 2: "D"}.get(int(action))
            row.append(action_symbol)
        strategy_map.append(row)

    df = pd.DataFrame(
        strategy_map, 
        columns=[f"D:{c}" for c in dealer_cards], 
        index=[f"P:{s}" for s in player_sums]
    )
    
    print(f"\n=== TABELA STRATEGII BOTA (True Count: {true_count}) ===")
    print("Analiza dla rąk bez użytecznego asa:")
    print(df)
    return df

print("\n--- TEST ZIMNEJ TALII (Brak wysokich kart) ---")
analyze_bot_strategy(model, true_count=-5.0)

print("\n--- TEST NEUTRALNEJ TALII (Początek buta) ---")
analyze_bot_strategy(model, true_count=0.0)

print("\n--- TEST GORĄCEJ TALII (Dużo wysokich kart) ---")
analyze_bot_strategy(model, true_count=5.0)


--- TEST ZIMNEJ TALII (Brak wysokich kart) ---

=== TABELA STRATEGII BOTA (True Count: -5.0) ===
Analiza dla rąk bez użytecznego asa:
     D:2 D:3 D:4 D:5 D:6 D:7 D:8 D:9 D:10 D:11
P:8    H   H   H   H   H   H   H   H    H    H
P:9    H   H   H   H   H   H   H   H    H    H
P:10   D   D   D   D   H   H   H   H    H    H
P:11   D   D   D   D   H   H   H   H    H    H
P:12   H   H   H   H   H   H   H   H    H    H
P:13   H   H   H   H   H   H   H   H    H    H
P:14   H   H   H   H   H   H   H   H    H    H
P:15   H   H   H   H   H   H   H   H    H    H
P:16   S   H   H   S   S   H   H   H    H    H
P:17   S   S   S   S   S   S   S   S    H    H
P:18   S   S   S   S   S   S   S   S    S    S
P:19   S   S   S   S   S   S   S   S    S    S

--- TEST NEUTRALNEJ TALII (Początek buta) ---

=== TABELA STRATEGII BOTA (True Count: 0.0) ===
Analiza dla rąk bez użytecznego asa:
     D:2 D:3 D:4 D:5 D:6 D:7 D:8 D:9 D:10 D:11
P:8    H   H   H   H   H   H   H   H    H    H
P:9    D   D   D   D   D   

,D:2,D:3,D:4,D:5,D:6,D:7,D:8,D:9,D:10,D:11
P:8,H,H,H,D,D,H,H,H,H,H
P:9,D,D,D,D,D,D,H,H,H,H
P:10,D,D,D,D,D,D,D,D,H,H
P:11,D,D,D,D,D,D,D,D,D,H
P:12,H,H,H,H,H,H,H,H,H,H
P:13,H,H,H,H,H,H,H,H,H,H
P:14,S,H,S,S,H,H,H,H,H,H
P:15,S,S,S,S,S,H,H,H,H,H
P:16,S,S,S,S,S,S,H,S,H,H
P:17,S,S,S,S,S,S,S,S,S,S
